# 02 · Exploring GSE300445 (primary melanoma before anti-PD1 immunotherapy, 10x Visium CytAssist)

Same task and structure as `01_explore_GSE316760.ipynb`: *explore the expression matrices, spatial coordinates,
metadata and tissue images; check coordinate units and available annotations.*

**The study** ([PMID 42315253](https://pubmed.ncbi.nlm.nih.gov/42315253/)): primary cutaneous melanomas from
stage III patients, sampled **before** adjuvant anti-PD1 immunotherapy. The authors map melanoma cell states and
their immune neighbourhoods (especially macrophages) and relate them to treatment efficacy, so this dataset is
directly about *immune cells near tumour cells*, i.e. our research question.

### What is different from GSE316760

| | GSE316760 (notebook 01) | **GSE300445 (this notebook)** |
|---|---|---|
| Samples | 2 (mel2, mel3) | **4 patients** (AdjIII-0019, -0022, -0113, -0133) |
| Protocol | Visium FFPE, tissue directly on the Visium slide | **Visium CytAssist**: probes transferred from a normal glass slide onto the Visium slide by an instrument |
| Space Ranger | 2.0.1 | **3.0.0** |
| Gene panel | 17,943 genes (probe set v1), no `MT-` genes | **18,085 genes (probe set v2), includes 11 `MT-` genes**, so % mitochondrial QC is possible |
| File format | `filtered_feature_bc_matrix.h5` | **`matrix.mtx.gz` + `barcodes` + `features`** (no chemistry metadata inside) |
| File names | `GSM…_mel2_…` | `GSM…_AdjIII-0019_processed_…` |
| Units | 65 µm rule and grid agree | **65 µm rule is ~7% wrong; only the grid method is reliable** (section 5) |
| Clinical info on GEO | none | none (response to therapy is only in the paper) |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)   # harmless deprecation notices from squidpy/anndata

# Project root = the spatial-reasoning folder (the one with pyproject.toml), wherever the notebook runs from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STAGE = ROOT / "01_data_exploration"
sys.path.insert(0, str(ROOT / "src"))
from space_mel.io import load_visium, read_series_matrix
from space_mel.spatial import units_table

GSE = "GSE300445"
RAW = ROOT / f"data/raw/{GSE}/files"
PROCESSED = ROOT / "data/processed"
FIGS = STAGE / f"results/{GSE}"
FIGS.mkdir(parents=True, exist_ok=True)
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

## 1. Load the samples

Same loader as notebook 01. The only change is the file prefix: these files carry an extra `_processed` in their
name (`GSM9060732_AdjIII-0019_processed_matrix.mtx.gz`).

⚠️ **GEO typo:** GEO's sample list says "ADJIII-0013", but the files of that sample (GSM9060734) are named
"AdjIII-0113". We use the file name.

In [ ]:
SAMPLES = {
    "AdjIII-0019": "GSM9060732_AdjIII-0019_processed",
    "AdjIII-0022": "GSM9060733_AdjIII-0022_processed",
    "AdjIII-0113": "GSM9060734_AdjIII-0113_processed",
    "AdjIII-0133": "GSM9060735_AdjIII-0133_processed",
}
adatas = {
    lib: load_visium(RAW, prefix, PROCESSED / f"visium/{GSE}_{lib}", library_id=lib)
    for lib, prefix in SAMPLES.items()
}
adatas["AdjIII-0019"]

## 2. What is inside each AnnData object

Same slots as in notebook 01 (`X`, `obs`, `var`, `obsm["spatial"]`, `uns["spatial"]`). One small difference:
`var` has no `genome` column, because the `.mtx` format does not store it (the `.h5` file did).

In [ ]:
for lib, a in adatas.items():
    X = a.X
    is_int = np.allclose(X.data[:1000], np.round(X.data[:1000]))
    print(f"{lib}: {a.n_obs} spots x {a.n_vars} genes | {type(X).__name__}, "
          f"{X.nnz / (a.n_obs * a.n_vars):.1%} non-zero | integer counts: {is_int}")

a = adatas["AdjIII-0019"]
display(a.obs.head())
display(a.var.head())

## 3. Metadata

With `.mtx` files the chemistry and Space Ranger version are not stored with the data, so we read them from
GEO's **series matrix** (`data/raw/GSE300445/metadata/`), a small text file with each sample's description.

In [ ]:
meta = read_series_matrix(ROOT / f"data/raw/{GSE}/metadata/{GSE}_series_matrix.txt.gz")
cols = ["geo_accession", "source_name_ch1", "characteristics_ch1", "extract_protocol_ch1", "data_processing"]
with pd.option_context("display.max_colwidth", 250):
    display(meta[cols])

In [ ]:
for lib, a in adatas.items():
    n_mt = a.var_names.str.startswith("MT-").sum()
    print(f"{lib}: in_tissue values {a.obs.in_tissue.unique()} | MT- genes: {n_mt} | "
          f"uns metadata: {a.uns['spatial'][lib]['metadata']}")
print("MT genes in the panel:", list(adatas["AdjIII-0019"].var_names[adatas["AdjIII-0019"].var_names.str.startswith("MT-")]))

**Reading this:**
- **CytAssist** protocol, Space Ranger **3.0.0**, probe set **v2** (18,085 genes). The H&E section is on an ordinary
  glass slide and the CytAssist instrument transfers the probes onto a 6.5 × 6.5 mm Visium capture area.
- Unlike notebook 01, the panel **contains 11 mitochondrial genes**, so we can compute % mitochondrial counts (a
  rough marker of damaged/low-quality spots).
- All spots have `in_tissue == 1` (filtered matrix), like GSE316760.
- No clinical annotation (e.g. responder / non-responder to anti-PD1) on GEO.

## 4. Tissue images

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax, (lib, a) in zip(axes, adatas.items()):
    img = a.uns["spatial"][lib]["images"]["hires"]
    ax.imshow(img)
    ax.set_title(f"{lib}: hires {img.shape[1]}x{img.shape[0]} px")
    ax.axis("off")
fig.savefig(FIGS / "tissue_images.png", bbox_inches="tight")

In [ ]:
sq.pl.spatial_scatter(adatas["AdjIII-0019"], library_id="AdjIII-0019", color=None, na_color="black", alpha=0.4,
                      title="AdjIII-0019: spots over the tissue", figsize=(6, 6))

As in notebook 01, spots only exist inside the 6.5 × 6.5 mm capture area and only where there is tissue
(see the discussion there). With CytAssist the image is a scan of the original glass slide, registered to the
capture area by Space Ranger.

## 5. Coordinate units: here the 65 µm rule fails

In notebook 01 two methods agreed: (1) spots are 100 µm apart, and (2) `65 / spot_diameter_fullres`.
Here we compare them again with the shared helper `space_mel.spatial.units_table`, which stores micrometre
coordinates in `obsm["spatial_um"]` using method (1).

In [ ]:
units = units_table(adatas)
units

**Conclusion:** the grid method gives ≈0.23–0.25 µm/px, but the 65 µm rule is **~7% off** in every sample.
Space Ranger 3.0 / CytAssist defines `spot_diameter_fullres` differently from Space Ranger 2.0, so that number
cannot be trusted across versions. The physical grid (100 µm between spot centres) is the same on every Visium
slide, so we **always use the grid method** (also written in `docs/datasets.md`). An extra check, fitting the full
grid → pixel transformation, gave the same values within 0.5%.

The pixel size also differs between samples (0.229 vs 0.252 µm/px), which is why all distances must be in µm.

## 6. Quality control

Same metrics as notebook 01, plus **`pct_counts_mt`** (share of counts from mitochondrial genes), now possible.

In [ ]:
for lib, a in adatas.items():
    a.var["mt"] = a.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(a, qc_vars=["mt"], inplace=True, percent_top=None)

qc = pd.concat({lib: a.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]] for lib, a in adatas.items()},
               names=["sample"]).reset_index(level=0)
display(qc.groupby("sample").median().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, ["total_counts", "n_genes_by_counts", "pct_counts_mt"]):
    for lib in adatas:
        ax.hist(qc.loc[qc["sample"] == lib, col], bins=80, alpha=0.5, label=lib)
    ax.set_xlabel(col)
axes[0].legend()
fig.savefig(FIGS / "qc_histograms.png", bbox_inches="tight")

In [ ]:
for lib, a in adatas.items():
    sq.pl.spatial_scatter(a, library_id=lib, color=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
                          img_alpha=0.5, figsize=(4.5, 4.5), save=FIGS / f"qc_spatial_{lib}.png")

## 7. Marker genes

Same markers as notebook 01, plus two that matter for melanoma biology:

| Group | Genes |
|---|---|
| Melanoma / melanocytes | `MLANA`, `PMEL`, `SOX10` |
| Melanoma vs benign melanocytes | `PRAME` (used by pathologists: high in melanoma, low in nevi) |
| Proliferation | `MKI67` |
| Immune (all leukocytes) | `PTPRC` (CD45) |
| T cells | `CD3E`, `CD8A` |
| Macrophages / antigen presentation | `CD68`, `CD163`, `CD74` |
| Stroma (fibroblasts) | `COL1A1` |
| Epidermis (keratinocytes) | `KRT14` |

In [ ]:
MARKERS = ["MLANA", "PMEL", "SOX10", "PRAME", "MKI67", "PTPRC", "CD3E", "CD8A", "CD68", "CD163", "CD74",
           "COL1A1", "KRT14"]

for lib, a in adatas.items():
    a.layers["counts"] = a.X.copy()
    sc.pp.normalize_total(a, target_sum=1e4)
    sc.pp.log1p(a)
    sq.pl.spatial_scatter(a, library_id=lib, color=MARKERS, img=False, ncols=7, figsize=(3, 3),
                          save=FIGS / f"markers_{lib}.png")

## 8. A first data-driven segmentation (clustering)

Same procedure as notebook 01, per sample.

In [ ]:
for lib, a in adatas.items():
    sc.pp.highly_variable_genes(a, n_top_genes=2000, flavor="seurat_v3", layer="counts")
    sc.pp.pca(a, n_comps=30, mask_var="highly_variable")
    sc.pp.neighbors(a)
    sc.tl.leiden(a, resolution=0.5, flavor="igraph", n_iterations=2, key_added="leiden")
    sc.tl.umap(a)
    print(lib, a.obs.leiden.value_counts().sort_index().to_dict())

In [ ]:
for lib, a in adatas.items():
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    sc.pl.umap(a, color="leiden", ax=axes[0], show=False, title=f"{lib}: UMAP")
    sq.pl.spatial_scatter(a, library_id=lib, color="leiden", ax=axes[1], title=f"{lib}: clusters in space")
    fig.savefig(FIGS / f"clusters_{lib}.png", bbox_inches="tight")

In [ ]:
for lib, a in adatas.items():
    sc.pl.matrixplot(a, MARKERS, groupby="leiden", standard_scale="var", title=lib, cmap="viridis", show=False)
    plt.savefig(FIGS / f"cluster_markers_{lib}.png", bbox_inches="tight")
    plt.show()

In [ ]:
for lib, a in adatas.items():
    sc.tl.rank_genes_groups(a, "leiden", method="wilcoxon")
    print(f"--- {lib}: top 8 genes per cluster")
    display(pd.DataFrame(a.uns["rank_genes_groups"]["names"]).head(8))

## 9. Save

One compressed `.h5ad` per sample in `data/processed/` (gitignored), with raw counts in `layers["counts"]`,
QC metrics, µm coordinates and clusters.

In [ ]:
for lib, a in adatas.items():
    path = PROCESSED / f"{GSE}_{lib}.h5ad"
    a.write_h5ad(path, compression="gzip")
    print("saved", path.relative_to(ROOT), f"({path.stat().st_size / 1e6:.0f} MB)")

## 10. Summary and open questions

**What the data contains (task checklist)**
- **Expression matrices:** 4 patients, 2,145–4,115 spots × 18,085 genes; raw integer counts, already filtered to
  spots under tissue. Very deep (median ~27,000–37,000 counts/spot) except AdjIII-0113 (~6,700).
- **Metadata:** CytAssist, Space Ranger 3.0.0, probe set v2 with 11 `MT-` genes (low % MT everywhere).
  GEO typo: "ADJIII-0013" = files "AdjIII-0113". No response-to-therapy labels on GEO.
- **Spatial coordinates:** full-resolution pixels; **0.229–0.252 µm/px** from the grid. `65 / spot_diameter_fullres`
  is off by exactly −7.14% in every sample (Space Ranger 3.0 uses a 70 µm reference). µm in `obsm["spatial_um"]`.
- **Tissue images:** scan of the whole glass slide; the 6.5 mm capture window covers only part of each section.
- **Annotations:** none.

**First biological look** (marker genes and clusters; hypotheses to check)
- **AdjIII-0019:** concentrically zoned nodule: `SOX10`+ throughout but `MLANA`/`PMEL` only on the left and at the
  rim; undifferentiated core with hypoxic patches (VEGFA, ENO1/2); `PRAME`-low; immune-poor.
- **AdjIII-0022:** mostly differentiated, dome-shaped tumour; moderate `PRAME`; few immune cells.
- **AdjIII-0113:** small tumour strip next to **immune-rich stroma** (T cells, macrophages, a CCL19 lymphoid area,
  lymphatic vessels). Shallow data.
- **AdjIII-0133:** dedifferentiated tumour (`SOX10`+, `PRAME`+, `MLANA`/`PMEL`−) with hypoxic regions and broadly high
  `CD74`. **Cluster 7 is a straight stripe at the capture-area edge: probable technical artefact.**

**Open questions / next steps**
1. Get the patients' response to anti-PD1 from the paper's supplement.
2. Check AdjIII-0133's cluster 7 against the H&E before using it.
3. AdjIII-0113 (immune-rich) vs AdjIII-0019 (immune-poor, zoned) are contrasting cases for tumour–immune distances.